In [1]:
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [2]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [5]:
def load_rag_chain(chunks):
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # creating vector database
    vector_db = Chroma.from_texts(texts=chunks
                                 ,embedding=embedding_model)

    retriever = vector_db.as_retriever(search_kwargs={"k": 5})

    prompt = ChatPromptTemplate.from_template("""
    You are an honest assistant.
    You will accept PDF files and you will answer the question asked by the user appropriately.
    If you don't know the answer, just say you don't know. Don't try to make up an answer.
    
    Context:
    {context}
    
    Question:
    {question}
    """)
    llm = OllamaLLM(model="gpt-oss:20b-cloud")

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
    )

    return rag_chain
